# 09 — Elo Model v2: Official Eloratings.net Formula

Implements the exact formula from eloratings.net:
- Fixed K values by tournament type (60/50/40/30/20)
- Official goal difference multiplier
- No time decay, no confederation multipliers
- Pre-WC elite boosts for top teams in weaker confederations

Saves to `elo_ratings_v2.csv` and `elo_all_teams_v2.csv` — does NOT overwrite v1 files.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load data

In [ ]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Total matches: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

## 2. Tournament K values

Official eloratings.net K weights:
- 60 — World Cup finals
- 50 — Continental championship finals + major intercontinental tournaments
- 40 — World Cup & continental qualifiers + major tournaments
- 30 — All other tournaments
- 20 — Friendly matches

In [ ]:
K_VALUES = {
    # ── K=60: World Cup finals ─────────────────────────────────
    'FIFA World Cup': 60,

    # ── K=50: Continental finals + major intercontinental ──────
    'UEFA Euro': 50,
    'Copa América': 50,
    'African Cup of Nations': 50,
    'AFC Asian Cup': 50,
    'Gold Cup': 50,
    'Oceania Nations Cup': 50,
    'Confederations Cup': 50,
    'CONMEBOL-UEFA Cup of Champions': 50,

    # ── K=40: WC qualifiers + continental qualifiers + major ───
    'FIFA World Cup qualification': 40,
    'UEFA Euro qualification': 40,
    'African Cup of Nations qualification': 40,
    'AFC Asian Cup qualification': 40,
    'Copa América qualification': 40,
    'Gold Cup qualification': 40,
    'CONCACAF Nations League qualification': 40,
    'Oceania Nations Cup qualification': 40,
    'UEFA Nations League': 40,
    'CONCACAF Nations League': 40,
    'AFC Challenge Cup': 40,
    'AFC Challenge Cup qualification': 40,

    # ── K=30: All other tournaments ────────────────────────────
    'Gulf Cup': 30,
    'Arab Cup': 30,
    'Arab Cup qualification': 30,
    'WAFF Championship': 30,
    'AFF Championship': 30,
    'AFF Championship qualification': 30,
    'EAFF Championship': 30,
    'EAFF Championship qualification': 30,
    'SAFF Cup': 30,
    'CECAFA Cup': 30,
    'COSAFA Cup': 30,
    'COSAFA Cup qualification': 30,
    'CFU Caribbean Cup': 30,
    'CFU Caribbean Cup qualification': 30,
    'UNCAF Cup': 30,
    'Amílcar Cabral Cup': 30,
    'Baltic Cup': 30,
    'Island Games': 30,
    'Muratti Vase': 30,
    'King\'s Cup': 30,
    'Kirin Cup': 30,
    'Kirin Challenge Cup': 30,
    'Cyprus International Tournament': 30,
    'Malta International Tournament': 30,
    'USA Cup': 30,
    'Nehru Cup': 30,
    'Merdeka Tournament': 30,
    'Dynasty Cup': 30,
    'Korea Cup': 30,
    'Lunar New Year Cup': 30,
    'FIFA Series': 30,
    'CONCACAF Series': 30,
    'CONIFA World Football Cup': 30,
    'CONIFA World Football Cup qualification': 30,

    # ── K=20: Friendlies ───────────────────────────────────────
    'Friendly': 20,
}

DEFAULT_K = 30   # for any tournament not in the list

def get_k(tournament):
    return K_VALUES.get(tournament, DEFAULT_K)

print("K values set!")
print(f"Unique tournaments in data: {df['tournament'].nunique()}")

# Check for any unmapped tournaments
unmapped = [t for t in df['tournament'].unique() if t not in K_VALUES]
if unmapped:
    print(f"\nUnmapped tournaments (will use K={DEFAULT_K}):")
    for t in sorted(unmapped):
        print(f"  {t}")

## 3. Goal difference multiplier

Official formula:
- 1 goal margin: K × 1
- 2 goal margin: K × 1.5
- 3 goal margin: K × 1.75
- 4+ goal margin: K × (1.75 + (N-3)/8)

In [ ]:
def goal_diff_multiplier(goal_diff):
    """Official eloratings.net goal difference multiplier."""
    n = abs(goal_diff)
    if n <= 1:
        return 1.0
    elif n == 2:
        return 1.5
    elif n == 3:
        return 1.75
    else:
        return 1.75 + (n - 3) / 8

# Sanity check
print("Goal diff multipliers:")
for gd in [0, 1, 2, 3, 4, 5, 6, 7]:
    print(f"  {gd} goals: {goal_diff_multiplier(gd):.3f}")

## 4. Pre-WC elite boosts

In [ ]:
ELITE_BOOST = 30

elite_teams = {
    # CAF
    'Morocco':       [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    'Senegal':       [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    # AFC
    'Japan':         [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    'South Korea':   [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    # CONCACAF
    'Mexico':        [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    'United States': [2002, 2006, 2010, 2014, 2018, 2022, 2026],
    # OFC
    'New Zealand':   [2002, 2006, 2010],
}

wc_matches = df[df['tournament'] == 'FIFA World Cup'].copy()
wc_matches['year'] = wc_matches['date'].dt.year
wc_starts = wc_matches.groupby('year')['date'].min().to_dict()

print("WC start dates & boosts:")
for y, d in sorted(wc_starts.items()):
    teams = [t for t, yrs in elite_teams.items() if y in yrs]
    print(f"  {y}: {d.date()} → {teams}")

## 5. Elo system — official formula

In [ ]:
STARTING_ELO   = 1500
HOME_ADVANTAGE = 100

elo_ratings    = {}
boosts_applied = set()

def get_elo(team):
    if team not in elo_ratings:
        elo_ratings[team] = STARTING_ELO
    return elo_ratings[team]

def expected_score(ra, rb):
    """Official We formula."""
    return 1 / (1 + 10 ** ((rb - ra) / 400))

def apply_pre_wc_boosts(current_date):
    for wc_year, start_date in wc_starts.items():
        if current_date == start_date:
            for team, years in elite_teams.items():
                if wc_year in years and (team, wc_year) not in boosts_applied:
                    if team in elo_ratings:
                        elo_ratings[team] += ELITE_BOOST
                        boosts_applied.add((team, wc_year))
                        print(f"  +{ELITE_BOOST} → {team} (WC {wc_year})")

def update_elo(ht, at, hs, as_, tournament, neutral, date):
    he, ae = get_elo(ht), get_elo(at)

    # Home advantage — dr = rating diff + 100 for home team
    dr_home = (he + (0 if neutral else HOME_ADVANTAGE)) - ae
    dr_away = (ae) - (he + (0 if neutral else HOME_ADVANTAGE))

    we_home = expected_score(he + (0 if neutral else HOME_ADVANTAGE), ae)
    we_away = 1 - we_home

    # Actual result
    if hs > as_:   w_home, w_away = 1, 0
    elif hs < as_: w_home, w_away = 0, 1
    else:          w_home, w_away = 0.5, 0.5

    # K with goal difference multiplier
    k   = get_k(tournament)
    gdm = goal_diff_multiplier(abs(hs - as_))
    k_adj = k * gdm

    elo_ratings[ht] = he + k_adj * (w_home - we_home)
    elo_ratings[at] = ae + k_adj * (w_away - we_away)

print("Processing matches...")
print()

for _, row in df.iterrows():
    apply_pre_wc_boosts(row['date'])
    update_elo(
        row['home_team'], row['away_team'],
        row['home_score'], row['away_score'],
        row['tournament'], row['neutral'], row['date']
    )

print(f"Done. {len(elo_ratings)} teams rated.")
print(f"Elite boosts applied: {len(boosts_applied)}")

## 6. Normalize & view rankings

In [ ]:
all_elos     = pd.Series(list(elo_ratings.values()))
current_mean = all_elos.mean()
current_std  = all_elos.std()
TARGET_MEAN, TARGET_STD = 1500, 150

for team in elo_ratings:
    elo_ratings[team] = TARGET_MEAN + (elo_ratings[team] - current_mean) * (TARGET_STD / current_std)

elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
elo_df = elo_df.sort_values('elo', ascending=False).reset_index(drop=True)
elo_df.index += 1

print("Global top 20:")
print(elo_df.head(20).to_string())

## 7. WC 2026 teams & comparison with v1

In [ ]:
gs       = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\group_stages_clean.csv")
wc_teams = gs['nation'].tolist()

wc_elo = elo_df[elo_df['team'].isin(wc_teams)].copy()
wc_elo.index = range(1, len(wc_elo) + 1)

print(f"All {len(wc_elo)} WC 2026 teams ranked (v2 — eloratings formula):")
print(wc_elo.to_string())

In [ ]:
# Compare v1 vs v2
try:
    v1 = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_ratings.csv").rename(columns={'elo': 'elo_v1'})

    # Normalize v1 to same scale
    v1_mean, v1_std = v1['elo_v1'].mean(), v1['elo_v1'].std()
    v1['elo_v1'] = TARGET_MEAN + (v1['elo_v1'] - v1_mean) * (TARGET_STD / v1_std)

    comparison = wc_elo.merge(v1[['team', 'elo_v1']], on='team', how='left')
    comparison['delta'] = (comparison['elo'] - comparison['elo_v1']).round(1)
    comparison = comparison.sort_values('delta', ascending=False)

    print("Teams that gained most (v2 vs v1):")
    print(comparison[['team', 'elo_v1', 'elo', 'delta']].head(15).round(1).to_string(index=False))
    print("\nTeams that dropped most:")
    print(comparison[['team', 'elo_v1', 'elo', 'delta']].tail(10).round(1).to_string(index=False))
except FileNotFoundError:
    print("elo_ratings.csv not found — skipping comparison")

## 8. Save — separate files, does NOT overwrite v1

In [ ]:
wc_elo.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_ratings_v2.csv", index=False)
elo_df.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_all_teams_v2.csv", index=False)

print("Saved (v2 — eloratings formula):")
print("  elo_ratings_v2.csv      — WC 2026 teams")
print("  elo_all_teams_v2.csv    — all teams")
print("\nv1 files untouched:")
print("  elo_ratings.csv")
print("  elo_all_teams.csv")